## Imports and Paths

In [2]:
import pandas as pd
import numpy as np
from scipy import signal
from pathlib import Path


 
DATASET_PATH = Path(r"data/dataset")
HR_DATA_PATH = Path(r"data/HR_data_2.csv")


## EDA signal processing Funtions

In [3]:
# SIGNAL PROCESSING Functions

#Low pass filter to remove noise
def eda_low_pass_filter(eda_signal, cutoff_hz=1.0, order=4):
    #Cutoff 1 Hz removes noise while preserving EDA dynamics (from my biomedical  class)  
    #fs = 1/T T=4 because there are 4 samples each seccond
    fs=4.0  
    # Nyquist frequency = half the sampling rate
    nyquist = fs / 2
  
    normalized_cutoff = cutoff_hz / nyquist
    
    # Design Butterworth low-pass filter
    b, a = signal.butter(order, normalized_cutoff, btype='low')
    
    # Apply filter forward and backward so that there is zero phase distortion
    eda_filtered = signal.filtfilt(b, a, eda_signal)
    
    return eda_filtered

#smooths the signal using the values after and before for each value
def eda_smoothing_moving_average(eda_signal):
    #fs = 1/T T=4 because there are 4 samples each seccond
    fs=4.0 
    window_seconds=4
    window_samples = int(window_seconds * fs)  # 5 seconds x 4 Hz = 20 samples
    
    # np.ones(20)/20 creates an array of 20 values each equal to 1/20 with the 10 values
    # before and 10 values after the current values being smoothed
    eda_smoothed = np.convolve(
        eda_signal,
        np.ones(window_samples) / window_samples,
        mode='same'  # output same length as input
    )
    
    return eda_smoothed

#Compute the total signal power within a specific frequency band
def compute_band_power(signal_data, fs, lowcut, highcut):
    nperseg = min(len(signal_data), 256)
    freqs, psd = signal.welch(signal_data, fs=fs, nperseg=nperseg)
    band_mask  = (freqs >= lowcut) & (freqs <= highcut)
    if band_mask.sum() == 0:
        return 0.0
    return float(np.trapezoid(psd[band_mask], freqs[band_mask]))

#Get the slope of the signal in a set interval using 2 moving window
def rolling_window_slope(values, window_size):

    n_windows = len(values) // window_size
    if n_windows < 2:
        return 0.0
    window_means = [
        np.mean(values[i * window_size:(i + 1) * window_size])
        for i in range(n_windows)
    ]
    x     = np.arange(n_windows)
    slope = np.polyfit(x, window_means, 1)[0]
    return float(slope)

#function to extract all eda feutures
def extract_eda_features(eda_path):
    # Tonic Conductance (SCL - Skin Conductance Level): The baseline level of conductance over a period of time.
    #Phasic Conductance (SCR - Skin Conductance Response): Rapid changes in conductance due to specific stimuli (e.g., sudden noise or stressor).
    # Phasic Conductance (SCR) The signal we want
    fs = 4.0
    features = {}

    try:
        df  = pd.read_csv(eda_path, index_col=0)
        eda_raw = df['EDA'].dropna().values

        if len(eda_raw) < 20:
            raise ValueError("EDA signal too short")

        eda_filtered = eda_low_pass_filter(eda_raw, cutoff_hz=1.0)

        eda_smoothed = eda_smoothing_moving_average(eda_filtered)

        eda_tonic = eda_low_pass_filter(eda_smoothed, cutoff_hz=0.05)

        eda_phasic = eda_smoothed - eda_tonic

        # Frequency domain power 
        features['EDA_tonic_power']  = compute_band_power(eda_tonic,  fs, 0.001, 0.05)
        features['EDA_phasic_power'] = compute_band_power(eda_phasic, fs, 0.05,  0.25)

        tonic_p  = features['EDA_tonic_power']
        phasic_p = features['EDA_phasic_power']
        features['EDA_phasic_tonic_ratio'] = (
            phasic_p / tonic_p if tonic_p > 1e-10 else 0.0
        )

        # Rolling window slope
        # Positive slope = EDA building over the phase (sustained arousal)
        # Negative slope = EDA decreasing (returning to calm)
        window_60s = int(60 * fs)  
        features['EDA_rolling_slope']          = rolling_window_slope(eda_smoothed, window_60s)
        features['EDA_phasic_rolling_slope']   = rolling_window_slope(eda_phasic,   window_60s)

        # First half vs second half mean 
        mid = len(eda_smoothed) // 2
        features['EDA_first_half_mean']  = float(np.mean(eda_smoothed[:mid]))
        features['EDA_second_half_mean'] = float(np.mean(eda_smoothed[mid:]))
        features['EDA_half_diff']        = (
            features['EDA_second_half_mean'] - features['EDA_first_half_mean']
        )

        # Phasic peak count
        # Count stress response events in the phasic signal
        if eda_phasic.std() > 1e-10:
            threshold = eda_phasic.mean() + 0.5 * eda_phasic.std()
            peaks, _ = signal.find_peaks(
                eda_phasic,
                height=threshold,
                distance=int(fs * 2)  # minimum 2 seconds between peaks
            )
            features['EDA_phasic_peak_count_raw'] = len(peaks)
        else:
            features['EDA_phasic_peak_count_raw'] = 0

    except Exception as e:
        print(f"  EDA error ({eda_path}): {e}")
        for k in [
            'EDA_tonic_power', 'EDA_phasic_power', 'EDA_phasic_tonic_ratio',
            'EDA_rolling_slope', 'EDA_phasic_rolling_slope',
            'EDA_first_half_mean', 'EDA_second_half_mean', 'EDA_half_diff',
            'EDA_phasic_peak_count_raw'
        ]:
            features[k] = np.nan

    return features

## HR signal processing Funtions

In [4]:
#Functions to extract hear rate feutures
def extract_hr_features(hr_path):
    # HR is sampled at 1 Hz one value per second
    #fs = 1/T T=1 because there is 1 sample each seccond
    fs = 1.0
    features = {}

    try:
        df = pd.read_csv(hr_path, index_col=0)
        hr = df['HR'].dropna().values

        if len(hr) < 10:
            raise ValueError("HR signal too short")

        # Artifact removal impossible human heart rate values
        hr = hr.copy().astype(float)
        hr[hr < 30] = np.nan
        hr[hr > 200] = np.nan

        # Interpolate the removed and missing values
        hr = pd.Series(hr).interpolate(method='linear').values

        # Rolling slope
        window_60s = int(60 * fs)  # 60 seconds x 1 Hz = 60 samples
        features['HR_rolling_slope'] = rolling_window_slope(hr, window_60s)

        # First half vs second half
        mid = len(hr) // 2
        features['HR_first_half_mean']  = float(np.mean(hr[:mid]))
        features['HR_second_half_mean'] = float(np.mean(hr[mid:]))
        features['HR_half_diff']        = (
            features['HR_second_half_mean'] - features['HR_first_half_mean']
        )

        # HRV approximation
        # HR_diff_std removed corr=0.999 with rmssd_proxy (redundant)
        hr_diffs = np.diff(hr)
        features['HR_diff_rmssd_proxy'] = float(np.sqrt(np.mean(hr_diffs ** 2)))

        # Frequency domain
        # HR_LF_power removed corr=0.815 with HR_HF_power (redundant)
        features['HR_HF_power'] = compute_band_power(hr, fs, 0.15, 0.40)
        lf = compute_band_power(hr, fs, 0.04, 0.15)
        hf = features['HR_HF_power']
        features['HR_LF_HF_ratio'] = lf / hf if hf > 1e-10 else 0.0

    except Exception as e:
        print(f"  HR error ({hr_path}): {e}")
        for k in [
            'HR_rolling_slope',
            'HR_first_half_mean', 'HR_second_half_mean', 'HR_half_diff',
            'HR_diff_rmssd_proxy',
            'HR_HF_power', 'HR_LF_HF_ratio'
        ]:
            features[k] = np.nan

    return features


## Temperture signal processing Funtions

In [5]:
def extract_temp_features(temp_path):
    # TEMP is sampled at 4 Hz four values per second
    #fs = 1/T T=4 because there are 4 samples each seccond
    fs = 4.0
    features = {}

    try:
        df   = pd.read_csv(temp_path, index_col=0)
        temp = df['TEMP'].dropna().values

        if len(temp) < 10:
            raise ValueError("TEMP signal too short")

        # Artifact removal impossible temperature rate values
        temp = temp.copy().astype(float)
        temp[temp < 25] = np.nan
        temp[temp > 40] = np.nan

        # Interpolate the removed and missing values
        temp = pd.Series(temp).interpolate(method='linear').values

        # Rolling slope
        window_60s = int(60 * fs)  
        features['TEMP_rolling_slope'] = rolling_window_slope(temp, window_60s)

        # Half difference
        # TEMP_first_half_mean and TEMP_second_half_mean removed  corr=0.992
        mid = len(temp) // 2
        features['TEMP_half_diff'] = float(np.mean(temp[mid:]) - np.mean(temp[:mid]))

        # Initial slope
        first_60 = temp[:window_60s]
        if len(first_60) > 1:
            features['TEMP_initial_slope'] = float(
                np.polyfit(np.arange(len(first_60)), first_60, 1)[0]
            )
        else:
            features['TEMP_initial_slope'] = 0.0

        # Overall linear trend
        x = np.arange(len(temp))
        features['TEMP_overall_slope'] = float(
            np.polyfit(x, temp, 1)[0]
        )

    except Exception as e:
        print(f"  TEMP error ({temp_path}): {e}")
        for k in [
            'TEMP_rolling_slope', 'TEMP_half_diff',
            'TEMP_initial_slope', 'TEMP_overall_slope'
        ]:
            features[k] = np.nan

    return features

## Creating the New Dataset

In [6]:
#Main loop iteration over all 312 observations and processing them
def main():
    print("Loading HR_data_2.csv...")
    hr_data = pd.read_csv(HR_DATA_PATH, index_col=0)
    print(f"  {len(hr_data)} observations found")

    # the  Emotions columns to carry over from HR_data_2.csv
    emotion_cols = [
        'upset', 'hostile', 'alert', 'ashamed', 'inspired',
        'nervous', 'attentive', 'afraid', 'active', 'determined'
    ]

    all_features = []

    for counter, (idx, row) in enumerate(hr_data.iterrows()):
        raw_path = row['raw_data_path']

        # Build full path to the observation folder
        obs_folder = DATASET_PATH.parent / Path(raw_path)

        print(f"Processing [{counter+1}/{len(hr_data)}]: {raw_path}")

        # Paths to each signal file
        eda_path  = obs_folder / 'EDA.csv'
        hr_path   = obs_folder / 'HR.csv'
        temp_path = obs_folder / 'TEMP.csv'

        # Extract features from each signal using our functions
        eda_feats  = extract_eda_features(eda_path)
        hr_feats   = extract_hr_features(hr_path)
        temp_feats = extract_temp_features(temp_path)

        # Metadata from HR_data_2.csv
        obs_features = {
            'original_index': idx,
            'Individual':     row['Individual'],
            'Phase':          row['Phase'],
            'Round':          row['Round'],
            'Puzzler':        row['Puzzler'],
            'Cohort':         row['Cohort'],
            'Team_ID':        row['Team_ID'],
            'Frustrated':     row['Frustrated'],
        }

        # Add all emotion columns
        for col in emotion_cols:
            obs_features[col] = row[col] if col in row else np.nan

        # Add signal features
        obs_features.update(eda_feats)
        obs_features.update(hr_feats)
        obs_features.update(temp_feats)

        all_features.append(obs_features)

    # Save results
    results_df = pd.DataFrame(all_features)
    results_df.to_csv("features_temporal2.csv", index=False)

    print(f"\nDone Saved {len(results_df)} rows x {len(results_df.columns)} columns")





if __name__ == '__main__':
    main()

Loading HR_data_2.csv...
  312 observations found
Processing [1/312]: dataset/D1_4/ID_1/round_3/phase3
Processing [2/312]: dataset/D1_4/ID_1/round_3/phase2
Processing [3/312]: dataset/D1_4/ID_1/round_3/phase1
Processing [4/312]: dataset/D1_4/ID_1/round_2/phase3
Processing [5/312]: dataset/D1_4/ID_1/round_2/phase2
Processing [6/312]: dataset/D1_4/ID_1/round_2/phase1
Processing [7/312]: dataset/D1_4/ID_1/round_4/phase3
Processing [8/312]: dataset/D1_4/ID_1/round_4/phase2
Processing [9/312]: dataset/D1_4/ID_1/round_4/phase1
Processing [10/312]: dataset/D1_4/ID_1/round_1/phase3
Processing [11/312]: dataset/D1_4/ID_1/round_1/phase2
Processing [12/312]: dataset/D1_4/ID_1/round_1/phase1
Processing [13/312]: dataset/D1_4/ID_3/round_3/phase3
Processing [14/312]: dataset/D1_4/ID_3/round_3/phase2
Processing [15/312]: dataset/D1_4/ID_3/round_3/phase1
Processing [16/312]: dataset/D1_4/ID_3/round_2/phase3
Processing [17/312]: dataset/D1_4/ID_3/round_2/phase2
Processing [18/312]: dataset/D1_4/ID_3/ro